# B-mode imaging with a linear array (GUI4us in a notebook)

The same environment and displays as `run.py` next to this notebook (four transmit schemes:
`classical`, `pwi`, `sta`, `dwi`), shown **inside the notebook**: the live B-mode, the control
panel (voltage, TGC), start/freeze and capture. Captured frames are ordinary numpy arrays here.

Requirements: ARRUS (e.g. the release wheel for your platform) and `pip install "gui4us[jupyter]"`.
The us4R session belongs to this kernel: run the last cell (or restart the kernel) to release it.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from gui4us.model import StreamDataId
from run import SEQUENCES, create_gui   # run.py in this directory

print("available sequences:", SEQUENCES)

In [ ]:
# The session configuration of your system, the transmit scheme and the TX voltage [V].
SESSION_CFG = os.environ.get("ARRUS_SESSION_CFG", "us4r.prototxt")
SEQUENCE = "classical"
VOLTAGE = 5

In [ ]:
# Opens the us4R session (on GUI4us' environment thread) and uploads the scheme.
gui = create_gui(SESSION_CFG, sequence=SEQUENCE, voltage=VOLTAGE, view="none")
gui.start()

## Live view

Displays, control panel and actions -- the same components as the browser view.

In [ ]:
gui.widget()

## Data into variables

`capture` returns the raw pipeline output (dB), not the 8-bit display frames.

In [ ]:
frames = gui.capture(20, timeout=60)
bmode = gui.captured_array(output=0)      # (n_frames, z, x) [dB]
print(bmode.shape, bmode.dtype)

In [ ]:
# The extents come from the ARRUS metadata (the reconstruction grid) [m].
(z_min, z_max), (x_min, x_max) = gui.metadata.output(StreamDataId("default", 0)).extents
plt.figure(figsize=(6, 5))
plt.imshow(bmode[-1], cmap="gray", vmin=20, vmax=80,
           extent=np.array([x_min, x_max, z_max, z_min])*1e3)
plt.xlabel("OX [mm]"); plt.ylabel("OZ [mm]"); plt.title(f"B-mode ({SEQUENCE})")
plt.colorbar(label="dB");

## Settings from code

The same settings as the control panel.

In [ ]:
print([setting.name for setting in gui.settings])
gui.set("Voltage", 10)
gui.set("TGC", np.linspace(20, 54, 10))

In [ ]:
# Release the us4R (stops the acquisition, HV off, closes the session).
gui.close()